In [1]:
import torch
import numpy as np
import time
import math
import sys

import torch_geometric.transforms as T
# IEEE-CIS Fraud Detection
from torch_geometric.datasets import Planetoid, Amazon
from torch_geometric.transforms import RemoveIsolatedNodes
from torch_geometric.utils import subgraph

sys.path.insert(1, '/home/xilinx/jupyter_notebooks/sgrace_lib')

import config
import sgrace
from sgrace import (SGRACEState,
                    GATConv_SGRACE, GCNConv_SGRACE,
                    SAGEConv_SGRACE, SAGEGAT_SGRACE,
                    GINConv_SGRACE, TransformerConv_SGRACE,
                    Relu_SGRACE, Linear_SGRACE, calibrate_sgrace)

torch.manual_seed(3407)

# ── Run configuration ────────────────────────────────────────────────────────

calibrating = 0 # 1 to calibrate thresholds depending on data
training   = 1   # 1 = train + evaluate,  0 = evaluate only
preload    = 0    # 1 = load saved weights before training (fine-tune mode)
num_epochs = 200
batch_value = 2048
full_graph = 1

config.large_config = 1

if training == 1:
    config.instant_layer_count = 1
    config.profiling = 1
else:
    config.instant_layer_count = 3
    config.profiling = 1
    
#set quantization target 
if config.large_config == 1:
 config.h_qbits = 8
 config.h_qbitsl = 8
 config.w_qbits = 8
 config.w_qbitsl = 8
else:
 config.h_qbits = 1
 config.h_qbitsl = 4
 config.w_qbits = 1
 config.w_qbitsl = 4


# ── Model selection ──────────────────────────────────────────────────
# Set pynq_class to the _PYNQ class to instantiate:
#   "GAT" | "GCN" | "SAGE" | "SAGEGAT" | "GIN" | "Transformer"
#   or any custom class defined in the model cell.
#
# model_type is derived automatically from the instantiated model's
# layer types — no need to set it manually.


#pynq_class = "None"
pynq_class = "GCN"
#pynq_class = "GAT"

In [2]:
from torch_geometric.loader import DataLoader, NeighborLoader

def remove_zero_feature_nodes(data):
    """Remove nodes with all-zero features and re-index edges."""
    mask = (data.x != 0).any(dim=1)
    new_edge_index, new_edge_attr = subgraph(
        mask, data.edge_index, data.edge_attr, relabel_nodes=True)
    data.x          = data.x[mask]
    data.y          = data.y[mask]
    data.edge_index = new_edge_index
    data.edge_attr  = new_edge_attr
    return data

# ── Dataset selection ─────────────────────────────────────────────────────────
# Uncomment exactly one dataset_sel line:

dataset_sel = "Cora"
#dataset_sel = "Citeseer"
#dataset_sel = "Pubmed"
#dataset_sel = 'Photo'       # Amazon
#dataset_sel = 'Computers'   # Amazon

if dataset_sel in ("Cora", "Citeseer", "Pubmed"):
    dataset = Planetoid(root="data/Planetoid", name=dataset_sel, split="full")
else:
    dataset = Amazon(root="data/Amazon", name=dataset_sel)

data = dataset[0]

# Remove isolated / zero-feature nodes
transform = RemoveIsolatedNodes()
data      = transform(data)
data      = remove_zero_feature_nodes(data)

# Set train / test masks
data.train_mask = torch.zeros(data.num_nodes, dtype=torch.bool)
data.train_mask[:data.num_nodes - 1000] = 1
data.test_mask  = torch.zeros(data.num_nodes, dtype=torch.bool)
data.test_mask[data.num_nodes - 1000:data.num_nodes - 500] = 1

print(f'Dataset: {dataset}')
print(f'  Nodes: {data.num_nodes}  |  Edges: {data.num_edges}')
print(f'  Features: {dataset.num_features}  |  Classes: {dataset.num_classes}')
print(f'  Avg degree: {data.num_edges / data.num_nodes:.2f}')

average_node_degree = data.num_edges / data.num_nodes

# ── Data loader selection ─────────────────────────────────────────────────────
# Option 1 — standard DataLoader (whole graph as a single batch)

if (full_graph==1):
 train_loader = DataLoader([data], batch_size=batch_value, shuffle=True)
 test_loader  = DataLoader([data], batch_size=batch_value, shuffle=False)
else:
# Option 2 — NeighborLoader (mini-batch sampling, useful for large graphs)
# Uncomment to use instead of Option 1:
 train_loader = NeighborLoader(data, batch_size=batch_value,
                               num_neighbors=[30],
                               input_nodes=data.train_mask, shuffle=True)
 test_loader  = NeighborLoader(data, batch_size=batch_value,
                               num_neighbors=[30],
                               input_nodes=data.test_mask,  shuffle=False)

# Initialise SGRACE hardware
# Initialise hardware — model program is written in the model cell
# after the model is instantiated and derive_model_type() is called.

sample = next(iter(train_loader))
print(f"Number of nodes in this subgraph: {sample.num_nodes}")

#state = SGRACEState.init(dataset.num_classes, calib_file="cora_calib.json")

state = SGRACEState.init(dataset.num_classes)

Dataset: Cora()
  Nodes: 2708  |  Edges: 10556
  Features: 1433  |  Classes: 7
  Avg degree: 3.90
Number of nodes in this subgraph: 2708
scale fea
3
f_align is set to  0
f_alignl is set to  0
quantization_scale_adj
255.0
quantization_scale_fea
255.0
quantization_scale_w
254.0
internal quantization graph is:
8
internal quantization linear is:
8
deq factor
4.0631776391272885


In [3]:
from torch.nn import Linear
import torch.nn.functional as F
from torch.nn import LeakyReLU
from torch_geometric.nn import GATConv,GCNConv
from torch_geometric.nn import global_mean_pool
#from ogb.graphproppred.mol_encoder import AtomEncoder

class GCN(torch.nn.Module):
    def __init__(self, hidden_channels):
        super(GCN, self).__init__()
        #torch.manual_seed(12345)
        #self.emb = AtomEncoder(dataset.num_node_features)
        self.att = GCNConv(dataset.num_node_features, hidden_channels,bias=False)
        self.conv2 = GCNConv(hidden_channels, hidden_channels,bias=False)
        #self.conv3 = GCNConv(hidden_channels, 16)
        self.lin = Linear(hidden_channels, dataset.num_classes,bias=False)
        #self.lin2 = Linear(16, 16)
        #self.lin3 = Linear(16, dataset.num_classes)

    def forward(self, x, edge_index):
        # 1. Obtain node embeddings 
        x = x.float()
        #x = self.emb(x)
        rmult = time.time()
        
        #recorder = DataRecorder(rails['0V85'].power)
        #print('CPU forward kernel on')
        #with recorder.record(0.2): # Sample every 500 ms
        #  amult = time.time()
        #  for _ in range(10):
        x = self.att(x, edge_index)

        x = x.relu() 
        #dmult =  time.time()   
         #if (config.profiling == 1):
        #print(recorder.frame)
        #x = self.att(x, edge_index)
        #x = x.relu()        
        #lrelu = LeakyReLU(0.1)
        #x = lrelu(x)
        #if (config.profiling == 1):
        # print('conv1 layer timing : {:.5f}ms'.format(1000/1*(time.time() - rmult)))
        #rmult = time.time()
        x = self.conv2(x, edge_index)
        #if (config.profiling == 1):
        # print('conv2 layer timing : {:.5f}ms'.format(1000/1*(time.time() - rmult)))
        
        #x = x.relu()
        #x = self.conv3(x, edge_index)
        
        # 2. Readout layer
        #x = global_mean_pool(x, batch)  # [batch_size, hidden_channels]
        #x = x.relu()
        #x = lrelu(x)

        # 3. Apply a final classifier
        #x = F.dropout(x, p=0.5, training=self.training)
        x = F.dropout(x, p=0.5, training=self.training)
        #rmult = time.time()
        x = self.lin(x)
        if (config.profiling == 1):
         print('CPU model timing : {:.5f}ms'.format(1000/1*(time.time() - rmult)))
        #if (config.profiling == 1):
        # print('Linear layer timing : {:.5f}ms'.format(1000/1*(time.time() - rmult)))
        #x = self.lin2(x)
        #x = self.lin3(x)
        
 
        #return F.log_softmax(x, dim=1)
        
        return x


#model = GCN(hidden_channels=config.hidden_channels)
#print(model)

In [4]:
from torch.nn import LeakyReLU
import torch.nn.functional as F
import math
import pandas as pd
from torch_scatter import scatter_add
from torch_geometric.utils import add_remaining_self_loops, sort_edge_index, degree


def sym_norm2(edge_index, num_nodes, edge_weight=None, improved=False, dtype=None):
    """Symmetric normalisation with fill value derived from average node degree."""
    if edge_weight is None:
        edge_weight = torch.ones((edge_index.size(1),), dtype=dtype,
                                  device=edge_index.device)
    fill_value = math.trunc(math.log2(average_node_degree)) if not improved else 2
    edge_index, edge_weight = add_remaining_self_loops(
        edge_index, edge_weight, fill_value, num_nodes)
    edge_index, edge_weight = sort_edge_index(edge_index, edge_weight)
    row, col = edge_index
    deg = scatter_add(edge_weight, row, dim=0, dim_size=num_nodes)
    deg_inv_sqrt = deg.pow(-0.5)
    deg_inv_sqrt[deg_inv_sqrt == float('inf')] = 0
    return edge_index, deg_inv_sqrt[row] * edge_weight * deg_inv_sqrt[col]


# ---------------------------------------------------------------------------
# Shared forward-pass helper
# ---------------------------------------------------------------------------

def _prep_adj(x, edge_index):
    """Normalise adjacency and build sparse tensor. Returns (edge_index, norm, adj)."""
    edge_index, norm = sym_norm2(edge_index, x.size(0), improved=False)
    adj = torch.sparse_coo_tensor(edge_index, norm)
    return edge_index, norm, adj


def _prep_adj_gin(x, edge_index):
    """
    Raw un-normalised adjacency for exact GIN semantics.
    norm = ones — plain sum aggregation, no D^{-1/2} A D^{-1/2}.
    No self-loops added — the (1+ε)I term is handled by the
    hardware SAGE self-loop branch (X · (1+ε)W) independently.
    """
    norm = torch.ones(edge_index.size(1), dtype=torch.float32)
    adj  = torch.sparse_coo_tensor(edge_index, norm)
    return edge_index, norm, adj



# ---------------------------------------------------------------------------
# GAT_PYNQ  —  two GATConv_SGRACE layers + Linear_SGRACE classifier
# ---------------------------------------------------------------------------

class GAT_PYNQ(torch.nn.Module):

    def __init__(self, hidden_channels, head_count):
        super(GAT_PYNQ, self).__init__()
        print("GAT_PYNQ INIT")
        self.conv1 = GATConv_SGRACE(state, dataset.num_node_features,
                                    hidden_channels, head_count,
                                    dropout=0.1, alpha=0.2, concat=False)
        self.conv2 = GATConv_SGRACE(state, hidden_channels * head_count,
                                    hidden_channels, 1)
        self.reluh = Relu_SGRACE()
        self.lin   = Linear_SGRACE(state, hidden_channels, dataset.num_classes)

    def _weights(self):
        """Return (srelul_l1, weights_l2, weights_l3) for hardware dispatch."""
        if config.acc == 1:
            srelu = max(self.reluh.srelu.data.item(), 0.0)
            w2    = self.conv2.weight.data
            w3    = (self.lin.weight_linear.data if config.total_layer_count > 2
                     else self.lin.weight.data)
            return srelu, w2, w3
        return 0.0, None, None

    def forward(self, x, edge_index):
        if config.profiling == 1:
            ptime = time.time()

        edge_index, norm, adj = _prep_adj(x, edge_index)
        srelu, w2, w3 = self._weights()

        if config.profiling == 1:
            fmult = time.time()

        # layer 1 — sparse input
        x = self.conv1(0, 0, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L1 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        x = self.reluh(x)
        if config.profiling == 1:
            print("Relu time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # layer 2 — dense input
        x = self.conv2(0, 1, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L2 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # classifier
        x = F.dropout(x.float(), p=0.5, training=self.training)
        x = self.lin(0, 1, 0, x, edge_index, norm, adj,
                     0.0, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("Linear time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            print("Model time:  {:.5f}ms".format(1000 * (time.time() - ptime)))
        return x


# ---------------------------------------------------------------------------
# GCN_PYNQ  —  two GCNConv_SGRACE layers + Linear_SGRACE classifier
# ---------------------------------------------------------------------------

class GCN_PYNQ(torch.nn.Module):

    def __init__(self, hidden_channels):
        super(GCN_PYNQ, self).__init__()
        print("GCN_PYNQ INIT")
        self.conv1 = GCNConv_SGRACE(state, dataset.num_node_features,
                                    hidden_channels)
        self.conv2 = GCNConv_SGRACE(state, hidden_channels, hidden_channels)
        self.reluh = Relu_SGRACE()
        self.lin   = Linear_SGRACE(state, hidden_channels, dataset.num_classes)

    def _weights(self):
        if config.acc == 1:
            srelu = max(self.reluh.srelu.data.item(), 0.0)
            w2    = self.conv2.weight.data
            w3    = (self.lin.weight_linear.data if config.total_layer_count > 2
                     else self.lin.weight.data)
            return srelu, w2, w3
        return 0.0, None, None

    def forward(self, x, edge_index):
        if config.profiling == 1:
            ptime = time.time()

        edge_index, norm, adj = _prep_adj(x, edge_index)
        srelu, w2, w3 = self._weights()

        if config.profiling == 1:
            fmult = time.time()

        # layer 1 — sparse input
        x = self.conv1(0, 0, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L1 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        x = self.reluh(x)
        if config.profiling == 1:
            print("Relu time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # layer 2 — dense input
        x = self.conv2(0, 1, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L2 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # classifier
        x = F.dropout(x.float(), p=0.5, training=self.training)
        x = self.lin(0, 1, 0, x, edge_index, norm, adj,
                     0.0, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("Linear time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            print("Model time:  {:.5f}ms".format(1000 * (time.time() - ptime)))
        return x


# ---------------------------------------------------------------------------
# SAGE_PYNQ  —  two SAGEConv_SGRACE layers + Linear_SGRACE classifier
#
# SAGEConv forward: (stream, dense, relu, input, edge_index, norm, adj, weights_l2)
# The SAGE path adds a residual self-loop branch (weight_linear) internally,
# so it only needs the next graph-layer weight for chained dispatch.
# ---------------------------------------------------------------------------

class SAGE_PYNQ(torch.nn.Module):

    def __init__(self, hidden_channels):
        super(SAGE_PYNQ, self).__init__()
        print("SAGE_PYNQ INIT")
        self.conv1 = SAGEConv_SGRACE(state, dataset.num_node_features,
                                     hidden_channels)
        self.conv2 = SAGEConv_SGRACE(state, hidden_channels, hidden_channels)
        self.reluh = Relu_SGRACE()
        self.lin   = Linear_SGRACE(state, hidden_channels, dataset.num_classes)

    def _weights(self):
        if config.acc == 1:
            srelu = max(self.reluh.srelu.data.item(), 0.0)
            w2    = self.conv2.weight.data
            w3    = (self.lin.weight_linear.data if config.total_layer_count > 2
                     else self.lin.weight.data)
            return srelu, w2, w3
        return 0.0, None, None

    def forward(self, x, edge_index):
        if config.profiling == 1:
            ptime = time.time()

        edge_index, norm, adj = _prep_adj(x, edge_index)
        srelu, w2, w3 = self._weights()

        if config.profiling == 1:
            fmult = time.time()

        # layer 1 — sparse input
        x = self.conv1(0, 0, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L1 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        x = self.reluh(x)
        if config.profiling == 1:
            print("Relu time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # layer 2 — dense input
        x = self.conv2(0, 1, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L2 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # classifier
        x = F.dropout(x.float(), p=0.5, training=self.training)
        x = self.lin(0, 1, 0, x, edge_index, norm, adj,
                     0.0, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("Linear time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            print("Model time:  {:.5f}ms".format(1000 * (time.time() - ptime)))
        return x


# ---------------------------------------------------------------------------
# SAGEGAT_PYNQ  —  two SAGEGAT_SGRACE layers + Linear_SGRACE classifier
#
# SAGEGAT combines GraphSAGE residual branch with GAT attention scoring.
# Forward signature matches SAGEConv: (stream, dense, relu, input,
#                                      edge_index, norm, adj, weights_l2)
# ---------------------------------------------------------------------------

class SAGEGAT_PYNQ(torch.nn.Module):

    def __init__(self, hidden_channels, head_count):
        super(SAGEGAT_PYNQ, self).__init__()
        print("SAGEGAT_PYNQ INIT")
        self.conv1 = SAGEGAT_SGRACE(state, dataset.num_node_features,
                                    hidden_channels, head_count,
                                    dropout=0.1, alpha=0.2, concat=False)
        self.conv2 = SAGEGAT_SGRACE(state, hidden_channels * head_count,
                                    hidden_channels, 1)
        self.reluh = Relu_SGRACE()
        self.lin   = Linear_SGRACE(state, hidden_channels, dataset.num_classes)

    def _weights(self):
        if config.acc == 1:
            srelu = max(self.reluh.srelu.data.item(), 0.0)
            w2    = self.conv2.weight.data
            w3    = (self.lin.weight_linear.data if config.total_layer_count > 2
                     else self.lin.weight.data)
            return srelu, w2, w3
        return 0.0, None, None

    def forward(self, x, edge_index):
        if config.profiling == 1:
            ptime = time.time()

        edge_index, norm, adj = _prep_adj(x, edge_index)
        srelu, w2, w3 = self._weights()

        if config.profiling == 1:
            fmult = time.time()

        # layer 1 — sparse input
        x = self.conv1(0, 0, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L1 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        x = self.reluh(x)
        if config.profiling == 1:
            print("Relu time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # layer 2 — dense input
        x = self.conv2(0, 1, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L2 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # classifier
        x = F.dropout(x.float(), p=0.5, training=self.training)
        x = self.lin(0, 1, 0, x, edge_index, norm, adj,
                     0.0, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("Linear time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            print("Model time:  {:.5f}ms".format(1000 * (time.time() - ptime)))
        return x



# ---------------------------------------------------------------------------
# GIN_PYNQ  —  two GINConv_SGRACE layers + Linear_SGRACE classifier
#
# GIN forward: (input, edge_index, norm, adj, relu, dense, stream,
#               weights_l2, weights_l3, attention_l2, attention_l3)
# Note: GINConv __init__ signature is (in_features, out_features, state, eps, train_eps)
# ---------------------------------------------------------------------------

class GIN_PYNQ(torch.nn.Module):

    def __init__(self, hidden_channels, eps=0.0, train_eps=True):
        super(GIN_PYNQ, self).__init__()
        print("GIN_PYNQ INIT")
        self.conv1 = GINConv_SGRACE(dataset.num_node_features, hidden_channels,
                                    state, eps=eps, train_eps=train_eps)
        self.conv2 = GINConv_SGRACE(hidden_channels, hidden_channels,
                                    state, eps=eps, train_eps=train_eps)
        self.reluh = Relu_SGRACE()
        self.lin   = Linear_SGRACE(state, hidden_channels, dataset.num_classes)

    def _weights(self):
        if config.acc == 1:
            w2 = self.conv2.weight.data
            w3 = (self.lin.weight_linear.data if config.total_layer_count > 2
                  else self.lin.weight.data)
            return w2, w3
        return None, None

    def forward(self, x, edge_index):
        if config.profiling == 1:
            ptime = time.time()

        edge_index, norm, adj = _prep_adj_gin(x, edge_index)
        w2, w3 = self._weights()

        if config.profiling == 1:
            fmult = time.time()

        # layer 1 — sparse input, relu fused
        x = self.conv1(x, edge_index, norm, adj,
                       relu=1, dense=0, stream=0,
                       weights_l2=w2, weights_l3=w3,
                       attention_l2=None, attention_l3=None)
        if config.profiling == 1:
            print("L1 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        x = self.reluh(x)
        if config.profiling == 1:
            print("Relu time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # layer 2 — dense input
        x = self.conv2(x, edge_index, norm, adj,
                       relu=1, dense=1, stream=0,
                       weights_l2=w2, weights_l3=w3,
                       attention_l2=None, attention_l3=None)
        if config.profiling == 1:
            print("L2 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # classifier
        x = F.dropout(x.float(), p=0.5, training=self.training)
        x = self.lin(0, 1, 0, x, edge_index, norm, adj,
                     0.0, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("Linear time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            print("Model time:  {:.5f}ms".format(1000 * (time.time() - ptime)))
        return x


# ---------------------------------------------------------------------------
# Transformer_PYNQ  —  two TransformerConv_SGRACE layers + Linear_SGRACE
#
# TransformerConv forward: (input, edge_index, norm, adj, relu, dense, stream,
#                           weights_l2, weights_l3, attention_l2, attention_l3)
# Note: TransformerConv __init__ signature is
#       (in_features, out_features, state, nheads, beta, concat, exact_attn)
# Set exact_attn=True for exact scaled dot-product attention (slower, CPU).
# Set exact_attn=False (default) to use the hardware attention approximation.
# ---------------------------------------------------------------------------

class Transformer_PYNQ(torch.nn.Module):

    def __init__(self, hidden_channels, head_count=1,
                 beta=False, concat=False, exact_attn=False):
        super(Transformer_PYNQ, self).__init__()
        print("Transformer_PYNQ INIT")
        self.conv1 = TransformerConv_SGRACE(dataset.num_node_features,
                                            hidden_channels, state,
                                            nheads=head_count, beta=beta,
                                            concat=concat,
                                            exact_attn=exact_attn)
        out1 = hidden_channels * head_count if concat else hidden_channels
        self.conv2 = TransformerConv_SGRACE(out1, hidden_channels, state,
                                            nheads=1, beta=beta,
                                            concat=False,
                                            exact_attn=exact_attn)
        self.lin   = Linear_SGRACE(state, hidden_channels, dataset.num_classes)

    def _weights(self):
        if config.acc == 1:
            w2 = self.conv2.weight.data
            w3 = (self.lin.weight_linear.data if config.total_layer_count > 2
                  else self.lin.weight.data)
            return w2, w3
        return None, None

    def forward(self, x, edge_index):
        if config.profiling == 1:
            ptime = time.time()

        edge_index, norm, adj = _prep_adj(x, edge_index)
        w2, w3 = self._weights()

        if config.profiling == 1:
            fmult = time.time()

        # layer 1 — sparse input, no fused relu (transformer uses its own
        #            attention normalisation; relu applied separately below)
        x = self.conv1(x, edge_index, norm, adj,
                       relu=0, dense=0, stream=0,
                       weights_l2=w2, weights_l3=w3,
                       attention_l2=None, attention_l3=None)
        if config.profiling == 1:
            print("L1 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        x = torch.relu(x)
        if config.profiling == 1:
            print("Relu time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # layer 2 — dense input (transformer features are dense after layer 1)
        x = self.conv2(x, edge_index, norm, adj,
                       relu=0, dense=1, stream=0,
                       weights_l2=w2, weights_l3=w3,
                       attention_l2=None, attention_l3=None)
        if config.profiling == 1:
            print("L2 time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # classifier
        x = F.dropout(x.float(), p=0.5, training=self.training)
        x = self.lin(0, 1, 0, x, edge_index, norm, adj,
                     0.0, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("Linear time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            print("Model time:  {:.5f}ms".format(1000 * (time.time() - ptime)))
        return x


# ---------------------------------------------------------------------------
# CUSTOM_PYNQ  —  GAT layer 1 + GCN layer 2 + Linear classifier
# ---------------------------------------------------------------------------

class CUSTOM_PYNQ(torch.nn.Module):

    def __init__(self, hidden_channels, head_count):
        super(CUSTOM_PYNQ, self).__init__()
        print("CUSTOM_PYNQ INIT")
        self.conv1 = GATConv_SGRACE(state, dataset.num_node_features,
                                    hidden_channels, head_count,
                                    dropout=0.1, alpha=0.2, concat=False)
        self.conv2 = GCNConv_SGRACE(state, hidden_channels * head_count,
                                    hidden_channels)
        self.reluh = Relu_SGRACE()
        self.lin   = Linear_SGRACE(state, hidden_channels, dataset.num_classes)

    def _weights(self):
        if config.acc == 1:
            srelu = max(self.reluh.srelu.data.item(), 0.0)
            w2    = self.conv2.weight.data
            w3    = (self.lin.weight_linear.data if config.total_layer_count > 2
                     else self.lin.weight.data)
            return srelu, w2, w3
        return 0.0, None, None

    def forward(self, x, edge_index):
        if config.profiling == 1:
            ptime = time.time()

        edge_index, norm, adj = _prep_adj(x, edge_index)
        srelu, w2, w3 = self._weights()

        if config.profiling == 1:
            fmult = time.time()

        # layer 1 — GAT, sparse input
        x = self.conv1(0, 0, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L1 GAT time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        x = self.reluh(x)
        if config.profiling == 1:
            print("Relu time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # layer 2 — GCN, dense input
        x = self.conv2(0, 1, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L2 GCN time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # classifier
        x = F.dropout(x.float(), p=0.5, training=self.training)
        x = self.lin(0, 1, 0, x, edge_index, norm, adj,
                     0.0, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("Linear time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            print("Model time:  {:.5f}ms".format(1000 * (time.time() - ptime)))
        return x

# ---------------------------------------------------------------------------
# HOGT_PYNQ  —  High-Order Graph Transformer
#
# Layer 1: HighOrderGATConv_SGRACE with k=2 (2-hop neighbourhood attention)
# Layer 2: GATConv_SGRACE          with k=1 (standard 1-hop attention)
# Lin:     Linear_SGRACE classifier
#
# Layer 1 attends over all nodes reachable in 2 hops.  The k-th order
# adjacency is computed once on the first forward call and cached.
# ---------------------------------------------------------------------------

class HOGT_PYNQ(torch.nn.Module):

    def __init__(self, hidden_channels, head_count, k=2):
        super(HOGT_PYNQ, self).__init__()
        print(f"HOGT_PYNQ INIT (k={k})")
        self.conv1 = HighOrderGATConv_SGRACE(
                         dataset.num_node_features, hidden_channels, state,
                         k=k, nheads=head_count,
                         dropout=0.1, alpha=0.2, concat=False)
        self.conv2 = GATConv_SGRACE(state, hidden_channels * head_count,
                                    hidden_channels, 1)
        self.reluh = Relu_SGRACE()
        self.lin   = Linear_SGRACE(state, hidden_channels, dataset.num_classes)

    def _weights(self):
        if config.acc == 1:
            srelu = max(self.reluh.srelu.data.item(), 0.0)
            w2    = self.conv2.weight.data
            w3    = (self.lin.weight_linear.data if config.total_layer_count > 2
                     else self.lin.weight.data)
            return srelu, w2, w3
        return 0.0, None, None

    def forward(self, x, edge_index):
        if config.profiling == 1:
            ptime = time.time()

        edge_index, norm, adj = _prep_adj(x, edge_index)
        srelu, w2, w3 = self._weights()

        if config.profiling == 1:
            fmult = time.time()

        # layer 1 — high-order GAT (k-hop), sparse input
        x = self.conv1(0, 0, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L1 HOGT time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        x = self.reluh(x)
        if config.profiling == 1:
            print("Relu time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # layer 2 — standard GAT, dense input
        x = self.conv2(0, 1, 1, x, edge_index, norm, adj,
                       srelu, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("L2 GAT time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            fmult = time.time()

        # classifier
        x = F.dropout(x.float(), p=0.5, training=self.training)
        x = self.lin(0, 1, 0, x, edge_index, norm, adj,
                     0.0, w2, w3, w3, None, None)
        if config.profiling == 1:
            print("Linear time: {:.5f}ms".format(1000 * (time.time() - fmult)))
            print("Model time:  {:.5f}ms".format(1000 * (time.time() - ptime)))
        return x

# ---------------------------------------------------------------------------
# Select and instantiate model
# ---------------------------------------------------------------------------

# Derive the ISA instruction program from the instantiated model's
# layer types and write it to the hardware model buffer.
# Call state.set_model() after the model is built below.

if pynq_class == "None":
    model = GCN(config.hidden_channels)
elif pynq_class == "GAT":
    model = GAT_PYNQ(config.hidden_channels, config.head_count)
elif pynq_class == "GCN":
    model = GCN_PYNQ(config.hidden_channels)
elif pynq_class == "SAGE":
    model = SAGE_PYNQ(config.hidden_channels)
elif pynq_class == "SAGEGAT":
    model = SAGEGAT_PYNQ(config.hidden_channels, config.head_count)
elif pynq_class == "GIN":
    model = GIN_PYNQ(config.hidden_channels)
elif pynq_class == "Transformer":
    model = Transformer_PYNQ(config.hidden_channels, config.head_count)
elif pynq_class == "CUSTOM":
    model = CUSTOM_PYNQ(config.hidden_channels, config.head_count)
elif pynq_class == "HOGT":
    model = HOGT_PYNQ(config.hidden_channels, config.head_count)
else:
    raise ValueError(f"Unknown pynq_class: {pynq_class!r}")
    
    
if pynq_class != "None":

 # Map layer class → operator name
 _layer_to_op = {
    "GCNConv_SGRACE":             "GCN",
    "GATConv_SGRACE":             "GAT",
    "SAGEConv_SGRACE":            "SAGE",
    "SAGEGAT_SGRACE":             "SAGEGAT",
    "GINConv_SGRACE":             "GIN",
    "TransformerConv_SGRACE":     "Transformer",
    "HighOrderGATConv_SGRACE":    "GAT",
 }

 def derive_model_type(model):
    """Build the ISA descriptor list by inspecting the model's layers."""
    lc   = config.total_layer_count
    # Collect graph conv layers (conv1, conv2, ...) and the linear classifier
    layers = []
    for attr in [f"conv{i+1}" for i in range(lc - 1)] + ["lin"]:
        layer = getattr(model, attr, None)
        if layer is None:
            raise AttributeError(
                f"Model has no attribute {attr!r}. "
                "Ensure your custom class uses conv1, conv2, ..., lin.")
        layers.append(layer)
    descs = []
    for i, layer in enumerate(layers):
        cls_name  = type(layer).__name__
        is_first  = (i == 0)
        is_linear = cls_name == "Linear_SGRACE"
        op = "Linear" if is_linear else _layer_to_op.get(cls_name)
        if op is None:
            raise ValueError(
                f"Unknown layer class {cls_name!r} at position {i}. "
                "Add it to _layer_to_op.")
        descs.append({
            "op":        op,
            "relu":      0 if is_linear else (0 if i == len(layers) - 2 else 1),
            "dense_fea": 0 if is_first else 1,
            "dense_adj": 0,
        })
    return descs

 model_type = derive_model_type(model)
 state.set_model(model_type)

print(model)

GCN_PYNQ INIT
Model program updated: custom
model_buffer  (total_layer_count=3, instant_layer_count=1)
  Slot   Byte     Operator           Flags
  ------ -------- ------------------ ------------------------------
  [0]    0x10     GCN/GIN            relu
  [1]    0x02     GCN/GIN            dense_fea
  [2]    0x42     Linear             dense_fea
GCN_PYNQ(
  (conv1): GCNConv_SGRACE (1433 -> 64)
  (conv2): GCNConv_SGRACE (64 -> 64)
  (reluh): Relu_SGRACE()
  (lin): Linear_SGRACE (64 -> 7)
)


In [5]:
import torch.nn.functional as F

criterion = torch.nn.CrossEntropyLoss()

# ── Learning rate ─────────────────────────────────────────────────────────────
if preload == 1:
    lr = 0.0001
elif config.w_qbits > 0:
    lr = 0.001
else:
    lr = 0.1

if preload == 1:
    model_path = "models/model_" + dataset_sel + "_8_8.ptx"
    model.load_state_dict(torch.load(model_path), strict=True)
    print(f"Loaded weights from {model_path}")

optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=0.001)


def accuracy(preds, labels):
    return np.equal(preds, labels).sum() / labels.size


def train():
    model.train()
    subgraph_count = 0
    for batch in train_loader:
        #print(f"Number of nodes in this subgraph: {batch.num_nodes}")
        subgraph_count+=1
        out  = model(batch.x, batch.edge_index)
        #print(f"Number of nodes in this subgraph: {batch.num_nodes}")
        loss = criterion(out[batch.train_mask], batch.y[batch.train_mask])
        loss.backward()
        optimizer.step()
        optimizer.zero_grad()
    print(f"Number of subgraphs:", subgraph_count)

def evaluate(loader, split="test"):
    model.eval()
    preds_l, labels_l = [], []
    for batch in loader:
        out = model(batch.x, batch.edge_index)
        mask = batch.test_mask if split == "test" else batch.train_mask
        preds_l.append(out[mask].detach().numpy())
        labels_l.append(batch.y[mask].detach().numpy())
    preds = np.argmax(np.concatenate(preds_l), axis=1)
    return accuracy(preds, np.concatenate(labels_l))


if training == 1:
    
    if calibrating == 1:
    
     print("CALIBRATING")

     calibrate_sgrace(
     model,
     forward_fn=lambda: model(data.x, data.edge_index),
     calib_fraction=0.15,
     percentile=60.0,  
     out_file="cora_calib.json",
     )

     state.recalibrate("cora_calib.json")
        
    else:    
     print("CALIBRATING SKIPPED")
    
    print("TRAINING")
    
    best_acc, best_epoch = 0.0, 0
    for epoch in range(num_epochs):
        t0 = time.time()
        train()
        test_acc = evaluate(test_loader, "test")
        if test_acc > best_acc:
            best_acc   = test_acc
            best_epoch = epoch
            model_path = "models/model_" + dataset_sel + ".ptx"
            torch.save(model.state_dict(), model_path)
            print("best model saved")
        print(f'Epoch: {epoch:03d}  Test Acc: {test_acc:.4f}  Time: {time.time()-t0:.4f}s')
    print(f'\nBest accuracy: {best_acc:.4f} at epoch {best_epoch}')

CALIBRATING SKIPPED
TRAINING
Transpose time: 0.21553ms
Accelerator forward kernel 1-layer time: 2.42496ms
output_acc time: 0.56291ms
Forward function time: 10.27846ms
L1 time: 30.17426ms
Relu time: 3.17478ms
Transpose time: 0.09418ms
Accelerator forward kernel 1-layer time: 2.47526ms
output_acc time: 0.04864ms
Forward function time: 6.41918ms
L2 time: 8.85391ms
Transpose time: 0.10657ms
Accelerator forward kernel 1-layer time: 1.68347ms
output_acc time: 0.04554ms
Forward function time: 5.72348ms
Linear time: 23.48351ms
Model time:  76.92575ms
CPU backward: 0.00948s
CPU backward: 0.01892s
CPU backward: 0.17484s
Number of subgraphs: 1
Transpose time: 0.08917ms
Accelerator forward kernel 1-layer time: 2.37799ms
output_acc time: 0.65875ms
Forward function time: 9.19652ms
L1 time: 28.25522ms
Relu time: 3.08180ms
Transpose time: 0.08607ms
Accelerator forward kernel 1-layer time: 2.45476ms
output_acc time: 0.04888ms
Forward function time: 6.37603ms
L2 time: 9.01937ms
Transpose time: 0.09012ms

CPU backward: 0.17937s
Number of subgraphs: 1
Transpose time: 0.09012ms
Accelerator forward kernel 1-layer time: 2.37322ms
output_acc time: 0.66376ms
Forward function time: 9.18841ms
L1 time: 27.65512ms
Relu time: 3.04866ms
Transpose time: 0.08702ms
Accelerator forward kernel 1-layer time: 2.45070ms
output_acc time: 0.05269ms
Forward function time: 6.29067ms
L2 time: 8.51655ms
Transpose time: 0.09370ms
Accelerator forward kernel 1-layer time: 1.70088ms
output_acc time: 0.04363ms
Forward function time: 5.23496ms
Linear time: 11.17039ms
Model time:  60.17852ms
best model saved
Epoch: 007  Test Acc: 0.6480  Time: 0.4115s
Transpose time: 0.08774ms
Accelerator forward kernel 1-layer time: 2.35343ms
output_acc time: 0.65827ms
Forward function time: 9.22465ms
L1 time: 27.84443ms
Relu time: 2.99835ms
Transpose time: 0.08726ms
Accelerator forward kernel 1-layer time: 2.45571ms
output_acc time: 0.05031ms
Forward function time: 6.34670ms
L2 time: 8.54969ms
Transpose time: 0.10657ms
Accelerator fo

CPU backward: 0.01791s
CPU backward: 0.17707s
Number of subgraphs: 1
Transpose time: 0.08750ms
Accelerator forward kernel 1-layer time: 2.36273ms
output_acc time: 0.07653ms
Forward function time: 9.21273ms
L1 time: 28.09024ms
Relu time: 3.21364ms
Transpose time: 0.08726ms
Accelerator forward kernel 1-layer time: 2.45452ms
output_acc time: 0.04935ms
Forward function time: 6.28805ms
L2 time: 8.48818ms
Transpose time: 0.09179ms
Accelerator forward kernel 1-layer time: 1.66106ms
output_acc time: 0.04244ms
Forward function time: 5.16367ms
Linear time: 11.14774ms
Model time:  60.63795ms
Epoch: 014  Test Acc: 0.8000  Time: 0.4110s
Transpose time: 0.08774ms
Accelerator forward kernel 1-layer time: 2.35486ms
output_acc time: 0.07820ms
Forward function time: 9.28211ms
L1 time: 27.89593ms
Relu time: 3.02386ms
Transpose time: 0.08726ms
Accelerator forward kernel 1-layer time: 2.45166ms
output_acc time: 0.05102ms
Forward function time: 6.30903ms
L2 time: 8.48341ms
Transpose time: 0.10777ms
Accelera

CPU backward: 0.17159s
Number of subgraphs: 1
Transpose time: 0.08821ms
Accelerator forward kernel 1-layer time: 2.34771ms
output_acc time: 0.07582ms
Forward function time: 9.16481ms
L1 time: 29.10686ms
Relu time: 2.99716ms
Transpose time: 0.08774ms
Accelerator forward kernel 1-layer time: 2.45047ms
output_acc time: 0.04983ms
Forward function time: 6.28591ms
L2 time: 8.48222ms
Transpose time: 0.08988ms
Accelerator forward kernel 1-layer time: 1.64390ms
output_acc time: 0.04196ms
Forward function time: 5.12218ms
Linear time: 10.99634ms
Model time:  61.30075ms
best model saved
Epoch: 021  Test Acc: 0.8360  Time: 0.4055s
Transpose time: 0.08798ms
Accelerator forward kernel 1-layer time: 2.35486ms
output_acc time: 0.07582ms
Forward function time: 9.18436ms
L1 time: 28.15318ms
Relu time: 3.03960ms
Transpose time: 0.08917ms
Accelerator forward kernel 1-layer time: 2.44570ms
output_acc time: 0.05007ms
Forward function time: 6.29807ms
L2 time: 8.49104ms
Transpose time: 0.10753ms
Accelerator fo

CPU backward: 0.02641s
CPU backward: 0.17004s
Number of subgraphs: 1
Transpose time: 0.08917ms
Accelerator forward kernel 1-layer time: 2.37107ms
output_acc time: 0.07653ms
Forward function time: 9.24778ms
L1 time: 27.95243ms
Relu time: 3.01218ms
Transpose time: 0.08678ms
Accelerator forward kernel 1-layer time: 2.48075ms
output_acc time: 0.05031ms
Forward function time: 6.30760ms
L2 time: 8.50487ms
Transpose time: 0.09084ms
Accelerator forward kernel 1-layer time: 1.65033ms
output_acc time: 0.04244ms
Forward function time: 5.12409ms
Linear time: 11.00612ms
Model time:  60.12201ms
Epoch: 028  Test Acc: 0.8440  Time: 0.4026s
Transpose time: 0.08678ms
Accelerator forward kernel 1-layer time: 2.39182ms
output_acc time: 0.07606ms
Forward function time: 9.19724ms
L1 time: 27.68803ms
Relu time: 3.07798ms
Transpose time: 0.08655ms
Accelerator forward kernel 1-layer time: 2.44164ms
output_acc time: 0.04911ms
Forward function time: 6.29902ms
L2 time: 8.48269ms
Transpose time: 0.10586ms
Accelera

...

CPU backward: 0.17946s
Number of subgraphs: 1
Transpose time: 0.08917ms
Accelerator forward kernel 1-layer time: 2.36464ms
output_acc time: 0.07629ms
Forward function time: 9.29761ms
L1 time: 28.06330ms
Relu time: 3.04103ms
Transpose time: 0.08845ms
Accelerator forward kernel 1-layer time: 2.45786ms
output_acc time: 0.04864ms
Forward function time: 6.31404ms
L2 time: 8.46958ms
Transpose time: 0.08678ms
Accelerator forward kernel 1-layer time: 1.66249ms
output_acc time: 0.04101ms
Forward function time: 5.15437ms
Linear time: 11.03711ms
Model time:  69.81158ms
Epoch: 168  Test Acc: 0.8520  Time: 0.3954s
Transpose time: 0.08941ms
Accelerator forward kernel 1-layer time: 2.35057ms
output_acc time: 0.07534ms
Forward function time: 9.17983ms
L1 time: 28.42140ms
Relu time: 3.09634ms
Transpose time: 0.08655ms
Accelerator forward kernel 1-layer time: 2.46716ms
output_acc time: 0.05054ms
Forward function time: 6.27589ms
L2 time: 8.46362ms
Transpose time: 0.10657ms
Accelerator forward kernel 1-la

CPU backward: 0.01786s
CPU backward: 0.16992s
Number of subgraphs: 1
Transpose time: 0.08774ms
Accelerator forward kernel 1-layer time: 2.35081ms
output_acc time: 0.07629ms
Forward function time: 9.56988ms
L1 time: 28.51391ms
Relu time: 3.05104ms
Transpose time: 0.08774ms
Accelerator forward kernel 1-layer time: 2.45833ms
output_acc time: 0.05150ms
Forward function time: 6.31261ms
L2 time: 8.47602ms
Transpose time: 0.08917ms
Accelerator forward kernel 1-layer time: 1.67060ms
output_acc time: 0.04196ms
Forward function time: 5.15223ms
Linear time: 11.03950ms
Model time:  68.54844ms
Epoch: 175  Test Acc: 0.8460  Time: 0.4024s
Transpose time: 0.08655ms
Accelerator forward kernel 1-layer time: 2.36511ms
output_acc time: 0.07439ms
Forward function time: 9.14383ms
L1 time: 27.62866ms
Relu time: 3.04079ms
Transpose time: 0.08607ms
Accelerator forward kernel 1-layer time: 2.44927ms
output_acc time: 0.05031ms
Forward function time: 6.41704ms
L2 time: 8.59714ms
Transpose time: 0.10467ms
Accelera

CPU backward: 0.17831s
Number of subgraphs: 1
Transpose time: 0.08655ms
Accelerator forward kernel 1-layer time: 2.36201ms
output_acc time: 0.07534ms
Forward function time: 9.17959ms
L1 time: 28.28932ms
Relu time: 3.03531ms
Transpose time: 0.08535ms
Accelerator forward kernel 1-layer time: 2.45523ms
output_acc time: 0.04911ms
Forward function time: 6.25086ms
L2 time: 8.43716ms
Transpose time: 0.08750ms
Accelerator forward kernel 1-layer time: 1.65510ms
output_acc time: 0.04077ms
Forward function time: 5.10907ms
Linear time: 10.98752ms
Model time:  60.48703ms
Epoch: 182  Test Acc: 0.8500  Time: 0.3963s
Transpose time: 0.08607ms
Accelerator forward kernel 1-layer time: 2.34747ms
output_acc time: 0.07582ms
Forward function time: 9.14574ms
L1 time: 27.63510ms
Relu time: 2.97117ms
Transpose time: 0.08607ms
Accelerator forward kernel 1-layer time: 2.48504ms
output_acc time: 0.05007ms
Forward function time: 6.28805ms
L2 time: 8.48126ms
Transpose time: 0.10538ms
Accelerator forward kernel 1-la

CPU backward: 0.02727s
CPU backward: 0.17147s
Number of subgraphs: 1
Transpose time: 0.08750ms
Accelerator forward kernel 1-layer time: 2.36034ms
output_acc time: 0.07606ms
Forward function time: 9.57942ms
L1 time: 28.74827ms
Relu time: 3.10683ms
Transpose time: 0.08583ms
Accelerator forward kernel 1-layer time: 2.47526ms
output_acc time: 0.04959ms
Forward function time: 6.28829ms
L2 time: 8.48699ms
Transpose time: 0.08941ms
Accelerator forward kernel 1-layer time: 1.64914ms
output_acc time: 0.04220ms
Forward function time: 5.13697ms
Linear time: 11.02114ms
Model time:  70.47248ms
Epoch: 189  Test Acc: 0.8520  Time: 0.4059s
Transpose time: 0.08774ms
Accelerator forward kernel 1-layer time: 2.39801ms
output_acc time: 0.07534ms
Forward function time: 9.16672ms
L1 time: 27.69899ms
Relu time: 3.03936ms
Transpose time: 0.08607ms
Accelerator forward kernel 1-layer time: 2.45285ms
output_acc time: 0.04840ms
Forward function time: 6.31356ms
L2 time: 8.49748ms
Transpose time: 0.10562ms
Accelera

CPU backward: 0.17887s
Number of subgraphs: 1
Transpose time: 0.08750ms
Accelerator forward kernel 1-layer time: 2.36511ms
output_acc time: 0.07582ms
Forward function time: 9.16409ms
L1 time: 28.11885ms
Relu time: 3.11947ms
Transpose time: 0.08750ms
Accelerator forward kernel 1-layer time: 2.45595ms
output_acc time: 0.04768ms
Forward function time: 6.25563ms
L2 time: 8.44526ms
Transpose time: 0.08893ms
Accelerator forward kernel 1-layer time: 1.66011ms
output_acc time: 0.04053ms
Forward function time: 5.11384ms
Linear time: 10.98776ms
Model time:  60.33254ms
Epoch: 196  Test Acc: 0.8460  Time: 0.3951s
Transpose time: 0.11444ms
Accelerator forward kernel 1-layer time: 2.38466ms
output_acc time: 0.07653ms
Forward function time: 9.18007ms
L1 time: 28.21565ms
Relu time: 3.00789ms
Transpose time: 0.08440ms
Accelerator forward kernel 1-layer time: 2.44522ms
output_acc time: 0.06104ms
Forward function time: 6.28328ms
L2 time: 8.46171ms
Transpose time: 0.10633ms
Accelerator forward kernel 1-la

In [6]:
# Load best saved weights and run inference

#model_path = "models/model_" + dataset_sel + "_8_8.ptx"
#model_path = "models/model_" + dataset_sel + "_1_4.ptx"
model_path = "models/model_" + dataset_sel + "_" + str(config.w_qbits) + "_" + str(config.w_qbitsl) + "_32.ptx"
model.load_state_dict(torch.load(model_path), strict=False)
state.my_ip.register_map.load_weights = 1

print("INFERENCE ONLY")
for epoch in range(5):
    t0       = time.time()
    test_acc = evaluate(test_loader, "test")
    print(f'Epoch: {epoch:03d}  Test Acc: {test_acc:.4f}  Time: {time.time()-t0:.4f}s')

RuntimeError: Error(s) in loading state_dict for GCN_PYNQ:
	size mismatch for conv1.weight: copying a param with shape torch.Size([1433, 32]) from checkpoint, the shape in current model is torch.Size([1433, 64]).
	size mismatch for conv1.bias: copying a param with shape torch.Size([32]) from checkpoint, the shape in current model is torch.Size([64]).
	size mismatch for conv2.weight: copying a param with shape torch.Size([32, 32]) from checkpoint, the shape in current model is torch.Size([64, 64]).
	size mismatch for conv2.bias: copying a param with shape torch.Size([32]) from checkpoint, the shape in current model is torch.Size([64]).
	size mismatch for lin.weight_linear: copying a param with shape torch.Size([32, 7]) from checkpoint, the shape in current model is torch.Size([64, 7]).

In [ ]:
from matplotlib import pyplot as plt


def _plot_weight_hist(weights_tensor, title, w_qbits, w_s, w_z):
    """Quantise weights and plot their histogram."""
    mybins = list(range(-2**(w_qbits - 1), 2**(w_qbits - 1) + 2))
    y = sgrace.quantization_qbits(weights_tensor, w_s, w_z, w_qbits)
    y = y.reshape(1, -1)
    print(f"{title} sparsity:")
    sgrace.isSparse(y, y.shape[0], y.shape[1])
    print(f"  max weight: {torch.max(weights_tensor).item():.6f}")
    print(f"  min weight: {torch.min(weights_tensor).item():.6f}")
    plt.figure(figsize=(10, 5))
    plt.xlabel(title)
    plt.ylabel("Frequency")
    plt.hist(y.flatten(), mybins)
    plt.xticks(mybins[::max(1, len(mybins)//16)],
               horizontalalignment="right", fontsize=12, rotation=90)
    plt.tight_layout()
    plt.show()


# --- Layer 1 weights (graph conv) ---
_plot_weight_hist(model.conv1.weight.data,
                  "Weights L1", config.w_qbits, state.w_s, state.w_z)

# --- Layer 2 weights (graph conv) ---
_plot_weight_hist(model.conv2.weight.data,
                  "Weights L2", config.w_qbits, state.w_s2, state.w_z2)

# --- Linear classifier weights ---
mybins_l = list(range(-2**(config.w_qbitsl - 1), 2**(config.w_qbitsl - 1) + 2))
if config.total_layer_count == 3:
    lin_w = model.lin.weight_linear.data
    y_lin = sgrace.quantization_qbits(lin_w, state.w_sl, state.w_zl, config.w_qbitsl)
else:
    lin_w = model.lin.weight.data
    y_lin = lin_w
y_lin = y_lin.reshape(1, -1)
print("Linear layer sparsity:")
sgrace.isSparse(y_lin, y_lin.shape[0], y_lin.shape[1])
print(f"  max weight: {torch.max(lin_w).item():.6f}")
print(f"  min weight: {torch.min(lin_w).item():.6f}")
plt.figure(figsize=(10, 5))
plt.xlabel("Weights Linear")
plt.ylabel("Frequency")
plt.hist(y_lin.flatten(), mybins_l)
plt.xticks(mybins_l[::max(1, len(mybins_l)//16)],
           horizontalalignment="center", fontsize=12, rotation=90)
plt.tight_layout()
plt.show()

# --- Internal feature calibration values ---
print(f"MAX FEA INTERNAL VALUE linear no acc: {state.global_max_input:.6f}")
print(f"MIN FEA INTERNAL VALUE linear no acc: {state.global_min_input:.6f}")
print(f"MAX FEA INTERNAL VALUE layer 1:       {state.cur_max_fea:.6f}")
print(f"MAX FEA INTERNAL VALUE layer 2:       {state.cur_max_fea2:.6f}")